# Searching and Sorting Arrays


```{contents}
:local:
:depth: 2
```


```{index} Ask What the Answer Must Be
```

## Ask What the Answer Must Be


A service desk stores the ticket IDs waiting for review in an array. Three common questions sound similar but need different answers: *Is ticket 42 waiting at all?*, *Where is the first ticket 42?*, and *How many tickets have ID 42?* Before writing a loop, decide which question the caller is asking, because the loop's result and its stopping rule depend on it.

| Question | Result type | Loop stops |
| --- | --- | --- |
| Is the value present? | `bool` | At the first match, or at the end |
| Where is the first match? | An index, or `-1` for missing | At the first match, or at the end |
| How many matches? | A count | Only at the end |

Returning `-1` for a missing value is a convention called a *sentinel*: it is a value that cannot be a real index, so the caller can test for it. Index zero is a real position, so never treat zero as "not found". A search result is an index, not the stored value; the caller reads the value through that index only after checking that the index is not `-1`.

This section introduces three small algorithms and the library methods that do the same jobs. **Searching** finds a matching position; **sorting** rearranges values into an order.

| Tool | Input requirement | Result | Changes the array? |
| --- | --- | --- | --- |
| Linear search | None | First matching index, or `-1` | No |
| Binary search | Sorted in ascending order | A matching index, or `-1` | No |
| Insertion sort | None | The same values in ascending order | Yes, in place |

The traversal and accumulator patterns from the previous section supply the loops; this section adds the stopping rules, the result contracts, and the one precondition that binary search cannot do without. Chapter 16 returns to these algorithms with costs, stability, comparers, and complete query workloads. The methods below assume the array reference is not `null`; an empty array is a valid input. For related explanation, see the further reading below.[^development-reading]


```{index} Linear Search: Visit Positions in Order
```

(ch07-linear-search)=

## Linear Search: Visit Positions in Order


A **linear search** checks each element in index order and returns as soon as it finds a match. It works on any array, sorted or not. The pattern is a `for` loop with an early `return`:[^ch07-linear]

```csharp
static int IndexOfFirst(int[] values, int target)
{
    for (int i = 0; i < values.Length; i++)
    {
        if (values[i] == target)
        {
            return i;      // first match: stop here
        }
    }
    return -1;             // every position checked, no match
}

int position = IndexOfFirst(ticketIds, 42);   // the call
```

`int[] values` receives the array reference, so no elements are copied; `target` receives the value to find. Reaching the `return -1` line proves something: the loop examined every index from `0` through `Length - 1` and none matched. For an empty array the loop body never runs, so the method returns `-1` immediately, which is the correct answer for "no first match".

Trace the search on the ticket IDs `42, 17, 42, 8`:

| Target | Index i | Value | Equal? | Action |
| --- | ---: | ---: | --- | --- |
| 8 | 0 | 42 | No | Advance |
| 8 | 1 | 17 | No | Advance |
| 8 | 2 | 42 | No | Advance |
| 8 | 3 | 8 | Yes | Return 3 and stop |
| 99 | 0 to 3 | all four | No | Loop ends; return -1 |

A match near the front costs one comparison; a missing value costs one comparison per element. Two mistakes are common. Placing `return -1` inside the loop reports "missing" after the first non-matching element, before the rest have been checked. Reading `values[position]` before testing `position >= 0` throws an exception when the value was missing. The library method `Array.IndexOf` implements the same first-match contract, so the example prints both results for comparison.


In [ ]:
using System;

int[] ticketIds = { 42, 17, 42, 8 };
Console.WriteLine($"First 42 at index {IndexOfFirst(ticketIds, 42)}");
Console.WriteLine($"First 8 at index {IndexOfFirst(ticketIds, 8)}");
Console.WriteLine($"First 99 at index {IndexOfFirst(ticketIds, 99)}");
Console.WriteLine($"Empty array: {IndexOfFirst(Array.Empty<int>(), 42)}");
Console.WriteLine($"Library Array.IndexOf: {Array.IndexOf(ticketIds, 42)}");

static int IndexOfFirst(int[] values, int target)
{
    for (int i = 0; i < values.Length; i++)
    {
        if (values[i] == target)
        {
            return i;
        }
    }
    return -1;
}


First 42 at index 0
First 8 at index 3
First 99 at index -1
Empty array: -1
Library Array.IndexOf: 0


The second `42` at index 2 is never reported, because the contract asks for the first match. If the caller needs every matching position, the loop must not return early; it collects indexes or counts them instead, as a later example shows.


```{index} Guided Practice: Change the Matching Rule
```

## Guided Practice: Change the Matching Rule


A warehouse wants the first bin whose stock quantity is *at least* a requested minimum. Keep the scan and the `-1` result; change only the comparison. Predict the answers before running: for quantities `2, 0, 7, 5` and minimum 5 the first qualifying index is 2, and for minimum 9 it is `-1`. Then test an empty array and a match at the final position. Finally, explain why a *last*-match variation must keep scanning after it finds a match.


In [ ]:
using System;

// Exercise: complete the first-at-least comparison.
Console.WriteLine(FirstAtLeast(new[] { 2, 0, 7, 5 }, 5));

static int FirstAtLeast(int[] values, int minimum)
{
    for (int i = 0; i < values.Length; i++)
    {
        // Your code starts here.
        // Return i when the current value meets the minimum.
        // Your code ends here.
    }
    return -1;
}


In [ ]:
using System;

// Solution: first quantity meeting a minimum.
Console.WriteLine(FirstAtLeast(new[] { 2, 0, 7, 5 }, 5));
Console.WriteLine(FirstAtLeast(new[] { 2, 0, 7, 5 }, 9));
Console.WriteLine(FirstAtLeast(Array.Empty<int>(), 5));
Console.WriteLine(FirstAtLeast(new[] { 2, 5 }, 5));

static int FirstAtLeast(int[] values, int minimum)
{
    for (int i = 0; i < values.Length; i++)
    {
        if (values[i] >= minimum) return i;
    }
    return -1;
}


2
-1
-1
1


The solution prints `2`, `-1`, `-1`, and `1`. Only the condition changed; the shape of the loop, the early return, and the sentinel are the same as in `IndexOfFirst`.


```{index} Count Matches and Find the Largest
```

## Count Matches and Find the Largest


Two more scans use the same traversal with a different result. Counting matches keeps an accumulator and never stops early. Finding the largest value keeps the *index* of the best value seen so far and replaces it only when a strictly larger value appears, so the first of several equal maximums wins.

```csharp
int bestIndex = 0;                         // assume position 0 is best so far
for (int i = 1; i < values.Length; i++)    // start at 1: position 0 is already counted
{
    if (values[i] > values[bestIndex]) bestIndex = i;
}
```

An empty array has no largest element, so the method must decide a policy for it. Returning `-1` keeps the sentinel convention; throwing an exception is another valid contract that Chapter 6 discussed. What is not acceptable is reading `values[0]` on an empty array.


In [ ]:
using System;

int[] dailySales = { 120, 340, 95, 340, 210 };
Console.WriteLine($"Days with sales of 340: {CountMatches(dailySales, 340)}");
int best = IndexOfLargest(dailySales);
Console.WriteLine($"Best day is index {best} with {dailySales[best]}");
Console.WriteLine($"Empty array: {IndexOfLargest(Array.Empty<int>())}");

static int CountMatches(int[] values, int target)
{
    int count = 0;
    foreach (int value in values)
    {
        if (value == target) count++;
    }
    return count;
}

static int IndexOfLargest(int[] values)
{
    if (values.Length == 0) return -1;
    int bestIndex = 0;
    for (int i = 1; i < values.Length; i++)
    {
        if (values[i] > values[bestIndex]) bestIndex = i;
    }
    return bestIndex;
}


Days with sales of 340: 2
Best day is index 1 with 340
Empty array: -1


Both 340 values are counted, but the best day is index 1, the first of them, because the comparison uses `>` rather than `>=`. Changing that one character changes which tied day is reported. Write the tie rule into the method's contract so the caller knows.


```{index} Sort with the Library, Then Search Faster
```

## Sort with the Library, Then Search Faster


Sorting puts elements in order under a comparison rule. .NET provides `Array.Sort`, which sorts the array *in place*: the array you pass is rearranged, and every alias of it sees the new order. If the original arrival order is itself information, sort a clone instead, exactly as the previous section advised for any method that may mutate its input.

```csharp
int[] sorted = (int[])arrivals.Clone();   // separate array, same values
Array.Sort(sorted);                        // rearranges sorted, not arrivals
int position = Array.BinarySearch(sorted, target);
if (position >= 0) { /* found at sorted[position] */ }
else               { /* missing */ }
```

`Array.BinarySearch` finds a value much faster than a scan, but it has a precondition: the array must already be sorted with the same comparison it will use. On an unsorted array it can report a value as missing even though it is present. It returns an index when the value is found and a negative number when it is not, so test the sign before using the result as an index. The negative value encodes where the value would belong; Chapter 16 shows how to decode it.


In [ ]:
using System;

int[] arrivals = { 60, 17, 42, 8, 42 };
int[] sorted = (int[])arrivals.Clone();
Array.Sort(sorted);
Console.WriteLine("Arrival order: " + string.Join(", ", arrivals));
Console.WriteLine("Sorted copy:   " + string.Join(", ", sorted));

int position = Array.BinarySearch(sorted, 17);
Console.WriteLine(position >= 0 ? $"17 found at sorted index {position}" : "17 missing");
position = Array.BinarySearch(sorted, 20);
Console.WriteLine(position >= 0 ? $"20 found at sorted index {position}" : "20 missing");


Arrival order: 60, 17, 42, 8, 42
Sorted copy:   8, 17, 42, 42, 60
17 found at sorted index 1
20 missing


The index 1 refers to the sorted copy, not to the arrival position of 17, which was index 1 only by coincidence here. Report positions from the array you searched.

Strings sort too, but the comparison rule matters. By default `Array.Sort` on strings uses the current culture's alphabetical rules; for product codes and IDs, pass `StringComparer.Ordinal` so the order depends only on character codes and is the same on every machine. Use the same comparer for the search that you used for the sort.


In [ ]:
using System;

string[] codes = { "SKU-200", "sku-010", "SKU-100" };
Array.Sort(codes, StringComparer.Ordinal);
Console.WriteLine(string.Join(", ", codes));
Console.WriteLine(Array.BinarySearch(codes, "SKU-100", StringComparer.Ordinal) >= 0 ? "SKU-100 stocked" : "SKU-100 missing");
Console.WriteLine(Array.BinarySearch(codes, "sku-100", StringComparer.Ordinal) >= 0 ? "sku-100 stocked" : "sku-100 missing");


SKU-100, SKU-200, sku-010
SKU-100 stocked
sku-100 missing


Ordinal order places every uppercase letter before every lowercase letter, so `sku-010` sorts last, and `sku-100` is reported missing because ordinal equality is case-sensitive. If the business treats codes as case-insensitive, use `StringComparer.OrdinalIgnoreCase` for both the sort and the search. Chapter 5 made the same point about string equality; a search is only as correct as its matching rule.


```{index} Why a Sorted Array Searches Faster
```

## Why a Sorted Array Searches Faster


A linear search cannot skip anything, because an unsorted array gives no reason to believe the target is on one side or the other. In a sorted array, one comparison with the middle element rules out half of the remaining positions. Searching `8, 17, 42, 42, 60` for 60:

| Remaining indexes | Middle index and value | Decision |
| --- | --- | --- |
| 0 through 4 | 2, value 42 | 42 < 60, discard indexes 0 through 2 |
| 3 through 4 | 3, value 42 | 42 < 60, discard index 3 |
| 4 | 4, value 60 | Found |

Each step halves the candidates, so the number of comparisons grows with the number of halvings, not with the number of elements:

| Elements | Linear search, missing value | Halving search, worst case |
| ---: | ---: | ---: |
| 10 | 10 | 4 |
| 1,000 | 1,000 | 10 |
| 1,000,000 | 1,000,000 | 20 |

Sorting costs work too, so the saving pays off when one sorted array answers many searches, such as a product catalog that is built once each morning and looked up all day. The next section writes the halving search by hand so you can see what `Array.BinarySearch` does.


```{index} Binary Search by Hand: Narrow a Sorted Interval
```

(ch07-binary-search)=

## Binary Search by Hand: Narrow a Sorted Interval


**Binary search** keeps track of the region that can still contain the target as two indexes. This book writes that region as a *half-open interval* `[low, high)`: `low` is the first candidate and `high` is one past the last candidate, so the region is empty exactly when `low == high`, and an empty array starts out empty because `high` is `0`. Chapter 16 uses the same convention.[^ch07-binary]

```csharp
static int BinarySearch(int[] values, int target)
{
    int low = 0;                        // first candidate index
    int high = values.Length;           // one past the last candidate
    while (low < high)                  // at least one candidate remains
    {
        int middle = low + (high - low) / 2;
        if (values[middle] == target) return middle;
        if (values[middle] < target) low = middle + 1;   // target is to the right
        else high = middle;                              // target is to the left
    }
    return -1;                          // candidate region is empty
}

int position = BinarySearch(sortedIds, wantedId);   // sortedIds must be ascending
```

Each pass compares the middle element with the target. Equal means found. A smaller middle means the target can only be to the right, so `low` moves past the middle. A larger middle means the target can only be to the left, so `high` moves down to the middle, which is excluded because `high` is one past the last candidate. Either way the region shrinks, so the loop ends. The expression `low + (high - low) / 2` picks the middle without adding two large indexes together.

Trace target 19 in `4, 8, 12, 19, 25`:

| low | high | middle | Value | Next action |
| ---: | ---: | ---: | ---: | --- |
| 0 | 5 | 2 | 12 | 12 < 19, set low to 3 |
| 3 | 5 | 4 | 25 | 25 > 19, set high to 4 |
| 3 | 4 | 3 | 19 | Found, return 3 |

For the absent target 20 the first two passes are the same; the third compares 19 < 20 and sets `low` to 4, which equals `high`, so the loop ends and the method returns `-1`. The runnable example prints `3`, `-1`, and `-1`.

Common mistakes: searching an array that is not sorted; starting `high` at `Length - 1` with this loop, which leaves the last element out of the region; writing `high = middle - 1`, which belongs to the other convention and skips a candidate here; and assuming the returned index is the *first* of several equal values, which this version does not promise. The index also refers to the sorted array, which may not be the arrival order.


In [ ]:
using System;

int[] ids = { 4, 8, 12, 19, 25 };
Console.WriteLine(BinarySearch(ids, 19));
Console.WriteLine(BinarySearch(ids, 20));
Console.WriteLine(BinarySearch(Array.Empty<int>(), 19));

static int BinarySearch(int[] values, int target)
{
    int low = 0;                        // first candidate index
    int high = values.Length;           // one past the last candidate
    while (low < high)                  // at least one candidate remains
    {
        int middle = low + (high - low) / 2;
        if (values[middle] == target) return middle;
        if (values[middle] < target) low = middle + 1;   // target is to the right
        else high = middle;                              // target is to the left
    }
    return -1;                          // candidate region is empty
}


3
-1
-1


```{index} Guided Practice: Repair the Boundary
```

## Guided Practice: Repair the Boundary


The starter mixes the two conventions: it starts `high` at `values.Length - 1` while keeping the half-open loop. Predict what it prints for the single-element array `12` with target 12 and for `4, 8, 12` with target 12 before running it. Repair the starting value, then check a first match, a last match, absence below and above the range, and empty input. Explain why passing all of those tests still does not remove the sorted-input precondition.


In [ ]:
using System;

// Exercise: intentional boundary defect. Expected after repair: 0 and 2.
Console.WriteLine(BinarySearch(new[] { 12 }, 12));
Console.WriteLine(BinarySearch(new[] { 4, 8, 12 }, 12));

static int BinarySearch(int[] values, int target)
{
    int low = 0;
    int high = values.Length - 1;   // Your code starts here: repair this starting value.
    while (low < high)
    {
        int middle = low + (high - low) / 2;
        if (values[middle] == target) return middle;
        if (values[middle] < target) low = middle + 1;
        else high = middle;
    }
    return -1;
}


In [ ]:
using System;

// Solution: high starts one past the last candidate.
Console.WriteLine(BinarySearch(new[] { 12 }, 12));
Console.WriteLine(BinarySearch(new[] { 4, 8, 12 }, 4));
Console.WriteLine(BinarySearch(new[] { 4, 8, 12 }, 12));
Console.WriteLine(BinarySearch(new[] { 4, 8, 12 }, 3));
Console.WriteLine(BinarySearch(new[] { 4, 8, 12 }, 13));
Console.WriteLine(BinarySearch(Array.Empty<int>(), 12));

static int BinarySearch(int[] values, int target)
{
    int low = 0;                        // first candidate index
    int high = values.Length;           // one past the last candidate
    while (low < high)                  // at least one candidate remains
    {
        int middle = low + (high - low) / 2;
        if (values[middle] == target) return middle;
        if (values[middle] < target) low = middle + 1;   // target is to the right
        else high = middle;                              // target is to the left
    }
    return -1;                          // candidate region is empty
}


0
0
2
-1
-1
-1


The defective starter prints `-1` twice: with `high` at `Length - 1`, the region `[0, 0)` for a single element is already empty, and in `4, 8, 12` the last element is never a candidate. The solution prints `0`, `0`, `2`, `-1`, `-1`, `-1`. The tests exercise the boundaries of a *sorted* array; none of them would detect that an unsorted array was passed, which is why the precondition belongs in the method's contract and not only in its tests.


```{index} Insertion Sort: Grow an Ordered Prefix
```

(ch07-insertion-sort)=

## Insertion Sort: Grow an Ordered Prefix


Library sorting is the normal choice in application code. Writing one small sorting algorithm shows what that call does. **Insertion sort** treats the first element as an already sorted group, then takes the next element, saves it, shifts the larger elements of the group one position to the right, and places the saved element in the opening, the way a clerk slots one more invoice into an ordered folder.[^ch07-sort]

```csharp
static void InsertionSort(int[] values)
{
    for (int i = 1; i < values.Length; i++)
    {
        int saved = values[i];                      // the element to insert
        int j = i - 1;
        while (j >= 0 && values[j] > saved)         // shift larger elements right
        {
            values[j + 1] = values[j];
            j--;
        }
        values[j + 1] = saved;                      // drop it into the opening
    }
}

InsertionSort(reportIds);                           // the call: the array itself changes
```

The method returns `void` because its result is the changed array, not a new one. Every alias sees the change, so if arrival order matters, clone the array first and sort the clone, as with `Array.Sort`. Trace `8, 4, 8, 2`:

| Pass | Saved value | Array afterward | Sorted prefix |
| --- | ---: | --- | --- |
| Start | | 8, 4, 8, 2 | 8 |
| i = 1 | 4 | 4, 8, 8, 2 | 4, 8 |
| i = 2 | 8 | 4, 8, 8, 2 | 4, 8, 8 |
| i = 3 | 2 | 2, 4, 8, 8 | whole array |

On the last pass, 2 is saved before 8, 8, and 4 shift right. During the shifts the array briefly holds repeated values; the saved local variable is what keeps the displaced element from being lost. `j` can reach `-1`, so the loop tests `j >= 0` before reading `values[j]`, and the opening `j + 1` can be index 0. Two mistakes are common: omitting the final `values[j + 1] = saved`, which loses a value, and sorting an alias while expecting the original to stay unchanged. Because the loop shifts only values *strictly greater* than the saved one, equal values keep their original relative order, a property Chapter 16 names stability and tests with labeled records.


In [ ]:
using System;

int[] original = { 8, 4, 8, 2 };
int[] report = (int[])original.Clone();
InsertionSort(report);
Console.WriteLine("Report:   " + string.Join(", ", report));
Console.WriteLine("Original: " + string.Join(", ", original));

static void InsertionSort(int[] values)
{
    for (int i = 1; i < values.Length; i++)
    {
        int saved = values[i];
        int j = i - 1;
        while (j >= 0 && values[j] > saved)
        {
            values[j + 1] = values[j];
            j--;
        }
        values[j + 1] = saved;
    }
}


Report:   2, 4, 8, 8
Original: 8, 4, 8, 2


Empty and single-element arrays need no insertion passes, and already sorted input still works, with one failed comparison per pass. Chapter 16 counts those comparisons for sorted, reversed, and duplicate inputs and shows why the library sort is faster on large arrays.


**Visualize it.** OpenDSA's [Insertion Sort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS2/html/InsertionSort.html) slideshow animates the same save, shift, and drop pass one comparison at a time, so you can check the trace above against it.

```{index} Guided Practice: Preserve the Saved Item
```

## Guided Practice: Preserve the Saved Item


Trace the incomplete starter on `3, 1, 3, 2` and say where each saved value belongs after the shifts. Restore the missing assignment, expect `1, 2, 3, 3`, and confirm that the length and the number of occurrences of each value are unchanged. Then test empty, single-element, and reverse-ordered arrays. A sort must preserve the values as well as establish the order.


In [ ]:
using System;

// Exercise: the insertion assignment is missing.
int[] values = { 3, 1, 3, 2 };
InsertionSort(values);
Console.WriteLine(string.Join(", ", values));

static void InsertionSort(int[] values)
{
    for (int i = 1; i < values.Length; i++)
    {
        int saved = values[i];
        int j = i - 1;
        while (j >= 0 && values[j] > saved)
        {
            values[j + 1] = values[j];
            j--;
        }
        // Your code starts here.
        // Put saved into the opening after shifting.
        // Your code ends here.
    }
}


In [ ]:
using System;

// Solution: retain each saved item after shifting.
int[] values = { 3, 1, 3, 2 };
InsertionSort(values);
Console.WriteLine(string.Join(", ", values));
int[] empty = Array.Empty<int>();
InsertionSort(empty);
Console.WriteLine($"Empty length: {empty.Length}");
int[] one = { 6 };
InsertionSort(one);
Console.WriteLine($"Single: {one[0]}");
int[] reverse = { 3, 2, 1 };
InsertionSort(reverse);
Console.WriteLine("Reversed input: " + string.Join(", ", reverse));

static void InsertionSort(int[] values)
{
    for (int i = 1; i < values.Length; i++)
    {
        int saved = values[i];
        int j = i - 1;
        while (j >= 0 && values[j] > saved)
        {
            values[j + 1] = values[j];
            j--;
        }
        values[j + 1] = saved;
    }
}


1, 2, 3, 3
Empty length: 0
Single: 6
Reversed input: 1, 2, 3


The starter prints `3, 3, 3, 3`: each shift copies a value to the right and nothing ever fills the opening, so the array fills with copies of its largest values. One assignment restores `1, 2, 3, 3`.


```{index} Choose Among the Three
```

## Choose Among the Three


| Situation | Reasonable choice | Why |
| --- | --- | --- |
| One lookup in arrival-order data | Linear search | No preparation; a scan costs at most one comparison per element |
| Many lookups in data that rarely changes | Sort once, then binary search | Sorting is paid once; each lookup needs only about log2 n comparisons |
| The order itself is the required output | A sort | Searching cannot produce an order |
| Data changes after almost every lookup | Linear search, or a different structure | Re-sorting after each change costs more than scanning |

Sorting solely to answer one small lookup is not a good trade, and binary search narrows candidates quickly only when the order already exists or has been prepared. For repeated lookups, count the preparation and the later updates as well as the lookups themselves. The algorithms in this section are instructional implementations; a library method has its own documented contract, and Chapter 16 develops the comparisons, duplicate policies, and workloads in full.


```{index} Guided Practice: Find a Product Code
```

## Guided Practice: Find a Product Code


Complete `IndexOfCode` so that `Report` prints the position of a stocked code and a clear message for a missing one. Use a linear search; the array is not sorted. Expected output:

```text
SKU-100 is stocked (position 1)
SKU-300 is not stocked
```


In [ ]:
using System;

string[] stocked = { "SKU-010", "SKU-100", "SKU-200" };
Report(stocked, "SKU-100");
Report(stocked, "SKU-300");

static int IndexOfCode(string[] codes, string target)
{
    // Your code starts here.
    return -1;
    // Your code ends here.
}

static void Report(string[] codes, string target)
{
    int index = IndexOfCode(codes, target);
    Console.WriteLine(index >= 0 ? $"{target} is stocked (position {index})" : $"{target} is not stocked");
}


In [ ]:
using System;

// Solution
string[] stocked = { "SKU-010", "SKU-100", "SKU-200" };
Report(stocked, "SKU-100");
Report(stocked, "SKU-300");

static int IndexOfCode(string[] codes, string target)
{
    for (int i = 0; i < codes.Length; i++)
    {
        if (codes[i] == target) return i;
    }
    return -1;
}

static void Report(string[] codes, string target)
{
    int index = IndexOfCode(codes, target);
    Console.WriteLine(index >= 0 ? $"{target} is stocked (position {index})" : $"{target} is not stocked");
}


SKU-100 is stocked (position 1)
SKU-300 is not stocked


Variation: change the matching rule to `string.Equals(codes[i], target, StringComparison.OrdinalIgnoreCase)` and search for `sku-100`. Then sort a clone of the array with `StringComparer.OrdinalIgnoreCase` and confirm that `Array.BinarySearch` with the same comparer agrees with your scan.


```{rubric} Footnotes
```


[^ch07-linear]: NIST Dictionary of Algorithms and Data Structures, [linear search](https://xlinux.nist.gov/dads/HTML/linearSearch.html). The first-match and -1 policies here are explicit lesson contracts.
[^ch07-binary]: NIST Dictionary of Algorithms and Data Structures, [binary search](https://xlinux.nist.gov/dads/HTML/binarySearch.html), including the sorted-input requirement and overflow-safe midpoint.
[^ch07-sort]: NIST Dictionary of Algorithms and Data Structures, [insertion sort](https://xlinux.nist.gov/dads/HTML/insertionSort.html). The lesson uses saved-item shifts within an integer array.


[^development-reading]: Further reading: [Harrington, Linear Searching](https://introcs.cs.luc.edu/arrays/linearsearch.html) and [Nakov and colleagues, Arrays](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-7-arrays/). The service-desk, warehouse, and inventory scenarios here are original teaching examples.
